<img src="https://ga-curriculum.github.io/universal-resources-internal/ga-logos/ga-red-cog-72x72.png" alt="" style="float: left; margin: 20px 20px 0 0; height: 55px">

# SQL IV: Introduction to SQlite and CRUD - PetDB API

_Author:_ Andres Gonzalez Flores

In this activity, you will use your knowledge of SQLite3 and python to build a functioning API for your pet database. You will be using the CRUD paradigm to interact with your database.


Learning Objectives:

- Carry out table creation, row insertion (C), reading (R), entry manipulation (U), and entry deletion (D) using SQL commands.
- Use  object-oriented programming (OOP) to create a full API that interacts with your SQL database

Prerequisites:
- You have completed the previous microlesson [petdb-create-crud-solutions.ipynb](./petdb-create-crud-solutions.ipynb)
- You are familiar with the CRUD paradigm
- you are familiar with object-oriented programming (OOP) in python
- you are familiar with basic sql commands like `CREATE TABLE`, `INSERT INTO`, `SELECT`, `UPDATE`, `DELETE`
- you are familiar with the pandas library

In [1]:
# Importing libraries
import sqlite3
import pandas as pd
#if you dont have the libraries installed, you can install them using pip (just uncomment the following lines)
# %pip install pandas
# %pip install sqlite3




Ok now that you are familiar with the CRUD paradigm (after completing the [petdb-create-crud-solutions.ipynb](./petdb-create-crud-solutions.ipynb) microlesson), let's use object-oriented programming (OOP) to create a simple API that interacts with your SQL database. 

**Refresher:** OOP is a programming paradigm that allows you to create objects that contain data and methods. In OOP, you define classes that encapsulate data and behavior, and then you create instances of those classes.

### The Scenario: "Paws & Claws Pet Adoption Insights"

Imagine you are working as a data analyst for "Paws & Claws," a large animal shelter that handles adoptions, care, and tracking of pets. They've recently upgraded their system and want to use data to improve their operations and adoption rates. Your goal is to use the PetDB to start exploring the pet data and provide insights to help the shelter. 

This PetDB exercise will walk you through setting up a sqlite database and capturing basic pet information (name, species, age, adoption_status). This is like creating a preliminary database for the shelter's system. The exercise will be broken into sections, each focusing on one or more aspects of CRUD. We'll use a combination of direct instructions and open-ended prompts.

You can start by creating a class called `PetDB` that will allow you to interact with your database. This class will have the following methods:
- `__init__`: this method will initialize the class and create the database connection and cursor.
- `create_pet_table`: this method will create the pets table.
- `insert_pet`: this method will insert a new pet into the database.
- `read_all_pets`: this method will return a pandas DataFrame with all the rows from the pets table.
- `pet_birthday`: this method will update the age of a pet by 1.
- `update_adoption_status`: this method will update the adoption_status of a pet.
- `drop_pet`: this method will delete a pet from the database.
  
Methods sound familiar? They are the same methods you used in the previous microlesson [petdb-create-crud-solutions.ipynb](./petdb-create-crud-solutions.ipynb). 

Now, let's create the class:

In [2]:
class PetDB:
    def __init__(self, file):
        self.conn = sqlite3.connect(file)
        self.cur = self.conn.cursor()
        self.create_pet_table()
    
    def create_pet_table(self):
        # Check if table exists first
        try:
            self.cur.execute("""
            CREATE TABLE pets (
            pet_name TEXT PRIMARY KEY NOT NULL,
            pet_age INTEGER,
            pet_species TEXT NOT NULL,
            adoption_status TEXT
            )
            """)
            self.conn.commit()
        except sqlite3.OperationalError:
            # Table already exists, so we can just pass
            pass
         
    # C
    def insert_pet(self, pet_name, pet_age, pet_species, adoption_status):
        try:
            insert_query = f'INSERT INTO pets VALUES ("{pet_name}", {pet_age}, "{pet_species}", "{adoption_status}")'
            self.cur.execute(insert_query)
            self.conn.commit()  # Don't forget to commit the changes!
        except sqlite3.IntegrityError:
            print(f"A pet named '{pet_name}' already exists in the database!")
        
    # R
    def read_all_pets(self):
        res = self.cur.execute("SELECT * FROM pets;")
        return pd.DataFrame(res.fetchall(), columns=['pet_name', 'pet_age', 'pet_species', 'adoption_status'])
    
    def get_query(self, query):
        '''Execute a custom query and return a pandas dataframe with the results'''
        res = self.cur.execute(query)
        columns = next(zip(*res.description))  # Get column names from cursor
        return pd.DataFrame(res.fetchall(), columns=columns)
    
    # U
    def pet_birthday(self, pet_name):
        birthday_query = f"""
        UPDATE pets
        SET pet_age = pet_age + 1
        WHERE pet_name = "{pet_name}"
        """

        self.cur.execute(birthday_query)
        self.conn.commit()
    
    # U
    def update_adoption_status(self, pet_name, adoption_status):
        update_query = f"""
        UPDATE pets
        SET adoption_status = "{adoption_status}"
        WHERE pet_name = "{pet_name}"
        """
        self.cur.execute(update_query)
        self.conn.commit()
        
    # D
    def drop_pet(self, pet_name):
        del_query = f'DELETE FROM pets WHERE pet_name = "{pet_name}"'
        self.cur.execute(del_query)
        self.conn.commit()


    def close_connection(self):
        self.conn.close()


Remember that when using sqlite3 in python we need to follow the following steps:
1. Connect to the database
2. Perform CRUD operations, depending on your tasks
3. Commit any changes to the database
4. Close the connection to the database after you are done

Closing the connection to the database is done to avoid memory leaks and to ensure that the database is properly closed and resources are released.

In [3]:
# connect to the database and create the your pet table
#for this we can use our class PetDB, by creating an instance of it we are creating a connection to the database called `paws_and_claws_db.db`
db = PetDB('paws_and_claws_db.db')

In [4]:
#you can check the database has been created in your working directory by running the following command:
!ls
# you can also check which tables are in the database by running the following command:
!sqlite3 paws_and_claws_db.db .tables

README.md                      paws_and_claws_db.db
assets                         petdb-api-crud-solutions.ipynb
pets


### Section 1: **Create** Operations    

Task 1.1: "Add the following pets to the database using the insert_pet method:

Pet Name: **'Buddy'**, Pet Species: **'Dog'**, Pet Age: **3**, Adaption Status: **'Adopted'**

Pet Name: **'Whiskers'**, Pet Species: **'Cat'**, Pet Age: **5**, Adaption Status: **'Adopted'** 

Pet Name: **'Tweety'**, Pet Species: **'Bird'**, Pet Age: **1**, Adaption Status: **'Pending'**

In [5]:
#insert the pets into the database according to the instructions
db.insert_pet('Buddy', 3, 'Dog', 'Adopted')
db.insert_pet('Whiskers', 5, 'Cat', 'Adopted')
db.insert_pet('Tweety', 1, 'Bird', 'Pending')

A pet named 'Tweety' already exists in the database!


In [6]:
#check if the pets have been inserted correctly by running the following command:
db.read_all_pets()

,pet_name,pet_age,pet_species,adoption_status
0,Tweety,1,Bird,Pending
1,Yukki,3,cat,Pending
2,Buddy,3,Dog,Adopted
3,Whiskers,5,Cat,Adopted


Task 1.2: Now let's add the some pets, who are not in the database yet.

* Name: 'Shelly', Species: 'Turtle', Age: 10, Owner: 'Alice'
* Name: 'Goldie', Species: 'Fish', Age: 2, Owner: 'Camille'


In [7]:
#insert new pets into the database
db.insert_pet('Shelly', 10, 'Turtle', 'Available')
db.insert_pet('Goldie', 2, 'Fish', 'Pending')

#check if the pets have been inserted correctly to our pets table:
db.read_all_pets()

,pet_name,pet_age,pet_species,adoption_status
0,Tweety,1,Bird,Pending
1,Yukki,3,cat,Pending
2,Buddy,3,Dog,Adopted
3,Whiskers,5,Cat,Adopted
4,Shelly,10,Turtle,Available
5,Goldie,2,Fish,Pending


**Stretch task 1.3:** Your colleague has provided you with a csv file with all the new pets that arrived to the shelter last month. You want to add them all to the database.

In [8]:

#read the csv file into a pandas dataframe
pet_data = pd.read_csv('./assets/pet_data.csv')
#insert the pets into the database
for index, row in pet_data.iterrows():
    db.insert_pet(row['pet_name'], row['pet_age'], row['pet_species'], row['adoption_status'])

#check if the pets have been inserted correctly to our pets table:  
db.read_all_pets()

A pet named 'Yukki' already exists in the database!


,pet_name,pet_age,pet_species,adoption_status
0,Tweety,1,Bird,Pending
1,Yukki,3,cat,Pending
2,Buddy,3,Dog,Adopted
3,Whiskers,5,Cat,Adopted
4,Shelly,10,Turtle,Available
5,Goldie,2,Fish,Pending
6,Georgie,12,cat,Adopted
7,Penelope,8,dog,Available
8,Leo,3,cat,Adopted
9,Lily,5,dog,Available


### Section 2: **Read** Operations

Task 2.1: Read all the pets from the database using the `read_all_pets` method and save them into a pandas dataframe called `pets_df`.
Then use the pandas dataframe to answer the following questions:

1. How many pets are there in the database?
2. What is the name and species, and age of the oldest pet?
3. How many pets are available for adoption?

In [9]:
#read all the pets from the database
pets_df = db.read_all_pets()

#How many pets are there in the database?
n_pets = pets_df.shape[0]
print(f"There are {n_pets} pets in the database")
#What is the name and species, and age of the oldest pet?
oldest_pet = pets_df.sort_values(by='pet_age', ascending=False).iloc[0]
print(f"The oldest pet is {oldest_pet['pet_name']}, a {oldest_pet['pet_species']} who is {oldest_pet['pet_age']} years old")
#How many pets are available for adoption?
n_available_pets = pets_df[pets_df['adoption_status'] == 'Available'].shape[0]
print(f"There are {n_available_pets} pets available for adoption")


There are 15 pets in the database
The oldest pet is Georgie, a cat who is 12 years old
There are 7 pets available for adoption


Stretch task 2.2: Use the `get_query` method to answer the same questions as above, namely:

1. How many pets are there in the database?
2. What is the name and species, and age of the oldest pet?
3. How many pets are available for adoption?

In [10]:
#1. How many pets are there in the database?
print(db.get_query("SELECT COUNT(*) FROM pets"))
#2. What is the name and species, and age of the oldest pet?
print(db.get_query("SELECT pet_name, pet_species, pet_age FROM pets ORDER BY pet_age DESC LIMIT 1"))
#3. How many pets are available for adoption?
print(db.get_query("SELECT COUNT(*) FROM pets WHERE adoption_status = 'Available'"))


   COUNT(*)
0        15
  pet_name pet_species  pet_age
0  Georgie         cat       12
   COUNT(*)
0         7


### Section 3: **Update** Operations

Task 3.1: Buddy, Goldie, and Zorro had birthdays last month Use the pet_birthday method to update their ages in your database. Then use the read_all_pets method to retrieve the updated data and print it."

In [11]:
#update the ages of the pets who had birthdays last month   
db.pet_birthday('Buddy')
db.pet_birthday('Goldie')
db.pet_birthday('Zorro')
#retrieve the updated data and print it
db.read_all_pets()

,pet_name,pet_age,pet_species,adoption_status
0,Tweety,1,Bird,Pending
1,Yukki,3,cat,Pending
2,Buddy,4,Dog,Adopted
3,Whiskers,5,Cat,Adopted
4,Shelly,10,Turtle,Available
5,Goldie,3,Fish,Pending
6,Georgie,12,cat,Adopted
7,Penelope,8,dog,Available
8,Leo,3,cat,Adopted
9,Lily,5,dog,Available


Task 3.2: "Penelope has been adopted! and there is someone interested in adopting River as well :), it is currently pending approval of their application. Use the update_adoption_status method to update their adoption status in your database. Then use the read_all_pets method to retrieve the updated data and print it."

In [12]:
#update the adoption status of Penelope and River
db.update_adoption_status('Penelope', 'Adopted')
db.update_adoption_status('River', 'Pending')
#retrieve the updated data and print it
db.read_all_pets()

,pet_name,pet_age,pet_species,adoption_status
0,Tweety,1,Bird,Pending
1,Yukki,3,cat,Pending
2,Buddy,4,Dog,Adopted
3,Whiskers,5,Cat,Adopted
4,Shelly,10,Turtle,Available
5,Goldie,3,Fish,Pending
6,Georgie,12,cat,Adopted
7,Penelope,8,dog,Adopted
8,Leo,3,cat,Adopted
9,Lily,5,dog,Available


### Section 4: **Delete** Operations

**Task 4.1:** You realized you made a mistake, and added goldie our fish mascot to the adoption database, but she is not a pet up for adoption! Use the drop_pet method to delete her from your database before someone attempts to adopt her. Then use the read_all_pets method to retrieve the updated data and print it."

In [13]:
#delete Goldie from the database
db.drop_pet('Goldie')
#retrieve the updated data and print it
db.read_all_pets()


,pet_name,pet_age,pet_species,adoption_status
0,Tweety,1,Bird,Pending
1,Yukki,3,cat,Pending
2,Buddy,4,Dog,Adopted
3,Whiskers,5,Cat,Adopted
4,Shelly,10,Turtle,Available
5,Georgie,12,cat,Adopted
6,Penelope,8,dog,Adopted
7,Leo,3,cat,Adopted
8,Lily,5,dog,Available
9,River,3,dog,Pending


Stretch Challenge: Task 4.2: This is a meant to be a challenging task, if you have completed all the previous tasks, you can try your hand at expanding the `PetDB` class to include a drop method that will delete a pet based on certain conditions (e.g. pet_name, pet_species, pet_age, adoption_status). 

You can then use this method to delete all the dogs from the database. and also delete all the pets that have already been adopted


In [14]:
# Extend the PetDB class with the new batch_delete method
class ExtendedPetDB(PetDB):
    def batch_delete(self, **conditions):
        """
        Delete multiple pets based on given conditions.
        
        Parameters:
            **conditions: Dict of column names and values to filter by
            Examples:
            - pet_species='Dog'
            - adoption_status='Available'
            - pet_age=5
        """
        try:
            # Build WHERE clause from conditions
            where_clauses = []
            for column, value in conditions.items():
                if isinstance(value, str):
                    where_clauses.append(f"{column} = '{value}'")
                else:
                    where_clauses.append(f"{column} = {value}")
            
            where_statement = " AND ".join(where_clauses) if where_clauses else "1=1"
            
            # Count pets to be deleted
            count_query = f"SELECT COUNT(*) FROM pets WHERE {where_statement}"
            count = self.get_query(count_query).iloc[0,0]
            
            # Execute deletion
            delete_query = f"DELETE FROM pets WHERE {where_statement}"
            self.cur.execute(delete_query)
            self.conn.commit()
            
            return f"Successfully deleted {count} pets"
            
        except Exception as e:
            return f"Error during batch deletion: {str(e)}"

# Create new instance with extended functionality
db = ExtendedPetDB('paws_and_claws_db.db')

# Test the new functionality
print("Deleting all dogs:")
print(db.batch_delete(pet_species='Dog'))

print("\nDeleting all available pets:")
print(db.batch_delete(adoption_status='Adopted'))

print("\nRemaining pets:")
print(db.read_all_pets())   

Deleting all dogs:
Successfully deleted 5 pets

Deleting all available pets:
Successfully deleted 4 pets

Remaining pets:
  pet_name  pet_age pet_species adoption_status
0   Tweety        1        Bird         Pending
1    Yukki        3         cat         Pending
2   Shelly       10      Turtle       Available
3   Steven        5       sloth       Available
4     Luna        3      rabbit       Available


## Recap

In this micro-lesson you have learned how to:

- Carried out table creation, row insertion (C), reading (R), entry manipulation (U), and entry deletion (D) using SQL commands.
- Used  object-oriented programming (OOP) to create a full API that interacts with your SQL database

You can now use this knowledge to interact with your SQL database and create your own APIs.

